In [1]:
using PlotlyJS
using Infiltrator
using XLSX
using CSV
include("./processing.jl")
include("../model/utils/empirical_mu.jl")


WebIO._IJuliaInit()

solve_empirical_μ_get_solution (generic function with 1 method)

In [2]:
day = 7
input_folder = "../input/base_case_increased_storage_energy_v8.0.8.4.1"
ε = 0.025
ρ = 0.8
gen_df, loads_multi_df, random_loads_multi_df, gen_variable_multi_df, storage_df, required_reserve = load_deterministic_data(day, input_folder, ε, ρ)
required_energy_reserve = load_energy_reserve(day, input_folder, loads_multi_df, gen_variable_multi_df, ε, ρ)
;

Reserve file found, loading reserves...
Energy reserve file found, loading reserves...


Checking that max_{i in T} ER[s,i,t]<=sum(R[S,τ], τ<=t) 

In [3]:
max_required_energy_reserve = combine(groupby(required_energy_reserve, [:t_hour]),[:reserve_up_MW, :reserve_down_MW].=> maximum, renamecols = false)
rename!(max_required_energy_reserve,:t_hour => :hour)
cumsum_required_reserve = combine(required_reserve, [:reserve_up_MW, :reserve_down_MW] .=> cumsum, renamecols = false)
# cumsum_required_reserve.hour .= required_reserve.hour
# cumsum_required_reserve = required_reserve[:, [:hour, :reserve_up_MW, :reserve_down_MW]] .|> cumsum 
delta_reserve = cumsum_required_reserve[:, [:reserve_up_MW, :reserve_down_MW]] .- max_required_energy_reserve[:, [:reserve_up_MW, :reserve_down_MW]]
delta_reserve.hour .= required_reserve.hour
all.(delta_reserve[:,[:reserve_up_MW, :reserve_down_MW]] .>=0)


Row,reserve_up_MW,reserve_down_MW
,Bool,Bool
1,true,true
2,true,true
3,true,true
4,true,true
5,true,true
6,true,true
7,true,true
8,true,true
9,true,true


In [4]:
s1 = scatter(cumsum_required_reserve, y = :reserve_up_MW)
s2 = scatter(max_required_energy_reserve, y = :reserve_up_MW)
p1 = plot([s1;s2])

s1 = scatter(cumsum_required_reserve, y = :reserve_down_MW)
s2 = scatter(max_required_energy_reserve, y = :reserve_down_MW)
p2 = plot([s1;s2])
[p1 p2]

# plot(stack(delta_reserve, [:reserve_up_MW, :reserve_down_MW]), x = :hour, y = :value, group = :variable)

data: [
  "scatter with fields type, xaxis, y, and yaxis",
  "scatter with fields type, xaxis, y, and yaxis",
  "scatter with fields type, xaxis, y, and yaxis",
  "scatter with fields type, xaxis, y, and yaxis"
]

layout: "layout with fields margin, template, xaxis1, xaxis2, yaxis1, and yaxis2"

In [5]:
cumsum_required_reserve = required_energy_reserve[:,[:i_hour,:t_hour]] # cumualtive required reserve between i and t
f(i,t,df) = 
    (reserve_up_MW = sum(df[(df.hour.>=i).&&(df.hour.<=t),:reserve_up_MW]),
    reserve_down_MW = sum(df[(df.hour.>=i).&&(df.hour.<=t),:reserve_down_MW]),
    )
transform!(groupby(cumsum_required_reserve,[:i_hour, :t_hour]), [:i_hour, :t_hour] => ((i,t) -> f(i,t,required_reserve)) => AsTable)
;
# delta with energy reserve. 
delta = cumsum_required_reserve[:, [:reserve_up_MW, :reserve_down_MW]] .- required_energy_reserve[:, [:reserve_up_MW, :reserve_down_MW]]
delta.i_hour .= cumsum_required_reserve.i_hour
delta.t_hour .= cumsum_required_reserve.t_hour
delta[delta.reserve_up_MW .>=0,:reserve_up_MW].= 0
delta[delta.reserve_down_MW .>=0,:reserve_down_MW].= 0
# delta = filter(row -> all(x -> x <= 0, row[[:reserve_up_MW, :reserve_down_MW]]), delta)
;
p1 = PlotlyJS.plot(heatmap(x=delta.t_hour, y=delta.i_hour, z=delta.reserve_up_MW, colorscale="Viridis"), Layout(title="cum reserve - energy reserve up MW", xaxis_title="t_hour", yaxis_title="i_hour"))
p2 = PlotlyJS.plot(heatmap(x=delta.t_hour, y=delta.i_hour, z=delta.reserve_down_MW, colorscale="Viridis"), Layout(title="cum reserve - energy reserve down MW", xaxis_title="t_hour", yaxis_title="i_hour"))
p3 = PlotlyJS.plot(heatmap(x=required_energy_reserve.t_hour, y=required_energy_reserve.i_hour, z=required_energy_reserve.reserve_up_MW, colorscale="Viridis"), Layout(title="energy energy reserve up MW", xaxis_title="t_hour", yaxis_title="i_hour"))
p4 = PlotlyJS.plot(heatmap(x=required_energy_reserve.t_hour, y=required_energy_reserve.i_hour, z=required_energy_reserve.reserve_down_MW, colorscale="Viridis"), Layout(title="energy reserve down MW", xaxis_title="t_hour", yaxis_title="i_hour"))
# p3 = PlotlyJS.plot(heatmap(x=sol_2.t, y=sol_2.i, z=sol_2.sDNDIS, colorscale="Viridis"))
# p4 = PlotlyJS.plot(heatmap(x=sol_2.t, y=sol_2.i, z=sol_2.sDNCH, colorscale="Viridis"))
[p1 p2; p3 p4]

data: [
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z",
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z",
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z",
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z"
]

layout: "layout with fields annotations, margin, template, xaxis1, xaxis2, xaxis3, xaxis4, yaxis1, yaxis2, yaxis3, and yaxis4"

In [6]:
temportal_weights = false
model, sol_1, sol_2 = solve_empirical_µ_get_solution(gen_df, loads_multi_df, storage_df, required_reserve, required_energy_reserve; temportal_weights = temportal_weights)
# sol_1 = dropmissing(sol_1)
;


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2475843
Academic license 2475843 - for non-commercial use only - registered to pa___@imperial.ac.uk
Set parameter MIPGap to value 1e-06
Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "Ubuntu 24.04.1 LTS")

CPU model: 11th Gen Intel(R) Core(TM) i7-11370H @ 3.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-06

Academic license 2475843 - for non-commercial use only - registered to pa___@imperial.ac.uk
Optimize a model with 3888 rows, 2932 columns and 18176 nonzeros
Model fingerprint: 0x00ee8c13
Model has 96 quadratic objective terms
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [1e+00, 1e+00]
  QObjective range [2e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 3e+03]
Presolve removed 1944 rows a

In [7]:
sol_1

Row,r_id,t,RESUP,RESDN,RESUPDIS,RESUPCH,RESDNCH,RESDNDIS,SOEUP,SOEDN,MAXERESUPDIS,MAXERESUPCH,MAXERESDNCH,MAXERESDNDIS,εUPDIS,εUPCH,εDNCH,εDNDIS,θUPDIS,θUPCH,θDNCH,θDNDIS
,Int64,Int64,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?
1,101,145,137.839,158.496,66.6727,71.1659,77.3574,81.139,0.0,0.0,66.6727,71.1659,77.3574,81.139,0.0,0.0,0.0,0.0,66.6727,71.1659,77.3574,81.139
2,101,146,151.751,187.473,75.0886,76.6627,92.9068,94.5666,0.0,0.0,140.91,146.951,168.289,173.594,0.0,0.0,0.0,0.0,74.2371,75.7855,90.9318,92.4546
3,101,147,119.137,154.442,56.7705,62.3669,74.4046,80.0376,0.0,0.0,196.611,208.237,240.36,251.215,0.0,0.0,0.0,0.0,55.7016,61.286,72.071,77.6212
4,101,148,148.309,172.043,72.8088,75.5,84.2495,87.7934,0.0,0.0,266.365,280.608,321.633,335.996,0.0,0.0,0.0,0.0,69.7532,72.3707,81.2731,84.7815
5,101,149,152.24,182.863,74.175,78.0652,90.21,92.6529,0.0,0.0,337.064,355.141,408.39,425.18,0.000202141,0.00019718,0.000171624,0.000176303,70.7001,74.5329,86.7573,89.1841
6,101,150,175.602,155.902,86.457,89.1454,74.5853,81.317,0.0,0.0,419.598,440.312,479.158,502.644,0.000198652,0.000193248,0.000203259,0.000208418,82.5335,85.171,70.7677,77.4642
7,101,151,195.462,215.94,96.8489,98.6131,107.69,108.25,0.0,0.0,511.935,534.345,582.521,606.548,0.000200427,0.000195182,0.000205618,0.000210403,92.3366,94.0328,103.363,103.903
8,101,152,201.691,215.163,100.103,101.588,106.475,108.688,0.0,0.0,606.763,630.499,684.16,710.299,0.000205936,0.00020154,0.000204087,0.000209138,94.8287,96.1544,101.639,103.751
9,101,153,163.316,161.752,78.0658,85.2504,76.1434,85.6091,0.0,0.0,678.359,708.891,754.638,789.95,0.000205347,0.000200696,0.000202985,0.000208355,71.5953,78.3921,70.4785,79.6506


In [8]:
println(value(model[:epsilon_variance]))
println(value(model[:theta]))
println(value(model[:reserve]))
# println(value(model[:slack]))
println(objective_value(model))

2.999091756235673e-6
5480.8830808012
9914.60860815
-4433.725524349713


In [9]:
function calculate_mu_t(sol_1_)
    sol_1 = dropmissing(sol_1_)
    mu = DataFrame()
    for (θ, RES) in [(:θUPDIS, :RESUPDIS), (:θUPCH, :RESUPCH), (:θDNDIS, :RESDNDIS), (:θDNCH, :RESDNCH)]
        # mu[!, Symbol("$θ/$RES")] = cumsum(sol_1[:, θ]) ./ cumsum(sol_1[:, RES]) # comment this to get cumulative ratio
        mu[!, Symbol("$θ/$RES")] = (sol_1[:, θ]) ./ (sol_1[:, RES])
    end
    mu[!, :t] = sol_1.t
    mu = mu[1:end, :]
    leftjoin!(sol_1, mu, on = :t)
    
    return unstack(stack(mu, Not(:t), variable_name=:mu),:t, :value)
end
mu_t = calculate_mu_t(sol_1)
# mu_t


Row,mu,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168
,String,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?
1,θUPDIS/RESUPDIS,1.0,0.98866,0.981172,0.958032,0.953152,0.954619,0.953408,0.947309,0.917114,0.920087,0.889093,0.841709,0.744235,0.47835,0.254096,0.123779,0.110407,0.068961,0.0570523,0.0327202,0.036074,0.0214216,0.0196907,0.0145539
2,θUPCH/RESUPCH,1.0,0.988558,0.982668,0.958553,0.954752,0.955416,0.953553,0.946516,0.91955,0.916032,0.887928,0.839483,0.740689,0.485205,0.224068,0.119765,0.108045,0.0749847,0.0521655,0.0321651,0.0351556,0.0202769,0.0175687,0.0116974
3,θDNDIS/RESDNDIS,1.0,0.977666,0.969808,0.965694,0.962561,0.95262,0.959845,0.954576,0.930399,0.941955,0.930146,0.897619,0.871638,0.83482,0.677804,0.402515,0.260943,0.125031,0.0965849,0.0573931,0.0452795,0.0371828,0.0324723,0.0246585
4,θDNCH/RESDNCH,1.0,0.978742,0.968636,0.964672,0.961726,0.948816,0.959815,0.954581,0.925603,0.945056,0.931516,0.896438,0.87153,0.78606,0.713509,0.418153,0.251644,0.129303,0.102158,0.0582346,0.0464173,0.0399352,0.0362776,0.0319941


In [10]:
ks = [:θUPDIS,:RESUPDIS]
s1 = scatter(stack(sol_1, ks), x = :t, y = :value, color=:r_id, group =:variable, facet_col = :r_id)
p1 = plot(s1)
# ks = ["θUPDIS/RESUPDIS"]
# s2 = scatter(stack(sol_1, ks),  x = :t, y = :value, color=:r_id, group =:variable, yaxis="y2")
# p1 = plot([s1; s2], Layout(yaxis2 = attr(overlaying="y", side="right")))

ks = [:θUPCH,:RESUPCH]
p2 = plot(stack(sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id)

ks = [:θDNDIS,:RESDNDIS]
p3 = plot(stack(sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id)

ks = [:θDNCH,:RESDNCH]
p4 = plot(stack(sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id)

[p1 p2; p3 p4]

┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257
┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257
┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257


data: [
  "scatter with fields color, facet, legendgroup, name, type, x, xaxis, y, and yaxis",
  "scatter with fields color, facet, legendgroup, name, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields margin, template, xaxis1, xaxis2, xaxis3, xaxis4, yaxis1, yaxis2, yaxis3, and yaxis4"

In [11]:
cumsum_sol_1 = combine(groupby(sol_1, :r_id), [:RESUPDIS, :RESDNDIS, :RESUPCH, :RESDNCH, :θUPDIS, :θUPCH, :θDNCH, :θDNDIS] .=> cumsum, renamecols = false)
cumsum_sol_1.t = unique(sol_1.t)
leftjoin!(cumsum_sol_1, sol_1[:, [:r_id, :t, :MAXERESUPDIS, :MAXERESUPCH, :MAXERESDNCH, :MAXERESDNDIS]], on = [:r_id, :t])

ks = [:θUPDIS,:RESUPDIS,:MAXERESUPDIS]
p1 = plot(stack(cumsum_sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id, title="Plot for $(ks[1]) and $(ks[2])")

ks = [:θUPCH,:RESUPCH, :MAXERESUPCH]
p2 = plot(stack(cumsum_sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id, title="Plot for $(ks[1]) and $(ks[2])")

ks = [:θDNDIS,:RESDNDIS, :MAXERESDNDIS]
p3 = plot(stack(cumsum_sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id, title="Plot for $(ks[1]) and $(ks[2])")

ks = [:θDNCH,:RESDNCH, :MAXERESDNCH]
p4 = plot(stack(cumsum_sol_1, ks), x = :t, y = :value, group=:r_id, color =:variable, facet_col = :r_id, title="Plot for $(ks[1]) and $(ks[2])")

[p1 p2; p3 p4]

┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257
┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257
┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257
┌ Warning: One of color or symbol present AND group -- group will be ignored
└ @ DataFramesExt ~/.julia/packages/PlotlyBase/YrV0o/ext/DataFramesExt.jl:257


data: [
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis",
  "scatter with fields legendgroup, marker, name, showlegend, title, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields margin, template, xaxis1, xaxis2, xaxis3, xaxis4, yaxis1, yaxis2, yaxis3, and yaxis4"

In [12]:
p1 = PlotlyJS.plot(heatmap(x=sol_2.t, y=sol_2.i, z=sol_2.ERESUPDIS, colorscale="Viridis"), Layout(title="ERESUPDIS"))
p2 = PlotlyJS.plot(heatmap(x=sol_2.t, y=sol_2.i, z=sol_2.ERESUPCH, colorscale="Viridis"), Layout(title="ERESUPCH"))
p3 = PlotlyJS.plot(heatmap(x=sol_2.t, y=sol_2.i, z=sol_2.ERESDNDIS, colorscale="Viridis"), Layout(title="ERESDNDIS"))
p4 = PlotlyJS.plot(heatmap(x=sol_2.t, y=sol_2.i, z=sol_2.ERESDNCH, colorscale="Viridis"), Layout(title="ERESDNCH"))
[p1 p2; p3 p4]

data: [
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z",
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z",
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z",
  "heatmap with fields colorscale, transpose, type, x, xaxis, y, yaxis, and z"
]

layout: "layout with fields annotations, margin, template, xaxis1, xaxis2, xaxis3, xaxis4, yaxis1, yaxis2, yaxis3, and yaxis4"

In [1]:
rhos = [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.99]
day = 7
temportal_weights = false
mu_t_all = DataFrame()
for rho in rhos
    input_folder = "../input/base_case_increased_storage_energy_v8.$(rho).4.1"
    gen_df, loads_multi_df, random_loads_multi_df, gen_variable_multi_df, storage_df, required_reserve = load_deterministic_data(day, input_folder)
    required_energy_reserve = load_energy_reserve(day, input_folder, loads_multi_df, gen_variable_multi_df)
    model, sol_1, sol_2 = solve_empirical_µ_get_solution(gen_df, loads_multi_df, storage_df, required_reserve, required_energy_reserve; temportal_weights = temportal_weights)
    mu_t = calculate_mu_t(sol_1)
    mu_t = insertcols(mu_t, 1, :rho => rho)
    append!(mu_t_all, mu_t)
end
CSV.write("empirical_mu.csv", mu_t_all)
# transform!(mu_t_all, AsTable(Not([:rho, :variable])) => ByRow(x -> mean(x)) => :mean)

LoadError: UndefVarError: `DataFrame` not defined in `Main`
Suggestion: check for spelling errors or missing imports.